In [ ]:
# Install all required libraries for the MCL Orchestrator project
!pip install fastapi uvicorn scikit-fem nest-asyncio requests pyngrok meshio -q

print("✅ All libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.2/166.2 kB 13.1 MB/s eta 0:00:00
✅ All libraries installed successfully


In [ ]:
import os
import json
import time
import logging
import threading
import requests
import nest_asyncio
from datetime import datetime

# Enable async behaviour in Colab
nest_asyncio.apply()

print("✅ Core libraries imported successfully")
print(f"   nest_asyncio applied — async enabled in Colab")

✅ Core libraries imported successfully
   nest_asyncio applied — async enabled in Colab


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Confirm mount was successful
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Google Drive mounted successfully")
    print(f"   Mounted at : /content/drive/MyDrive")
else:
    print("❌ Drive mount failed — re-run this cell and follow the authentication prompt")

Mounted at /content/drive
✅ Google Drive mounted successfully
   Mounted at : /content/drive/MyDrive


In [ ]:
# Master project path
DRIVE_PATH = '/content/drive/MyDrive/MCL_Orchestrator/'

# Individual subfolder paths
PATHS = {
    'notebooks' : f'{DRIVE_PATH}notebooks/',
    'services'  : f'{DRIVE_PATH}services/',
    'results'   : f'{DRIVE_PATH}results/',
    'logs'      : f'{DRIVE_PATH}logs/',
    'docs'      : f'{DRIVE_PATH}docs/'
}

# Confirm each folder exists
print("Confirming Drive folder structure:")
print("=" * 45)
all_exist = True
for name, path in PATHS.items():
    exists = os.path.exists(path)
    status = "✅ Found" if exists else "❌ Missing — create this folder in Drive"
    print(f"  {name:<12} : {status}")
    if not exists:
        all_exist = False

print("=" * 45)
print(f"  Base path : {DRIVE_PATH}")

if all_exist:
    print("\n✅ All folders confirmed — DRIVE_PATH is locked in")
else:
    print("\n⚠️  Some folders are missing — create them in Drive and re-run this cell")

Confirming Drive folder structure:
  notebooks    : ✅ Found
  services     : ✅ Found
  results      : ✅ Found
  logs         : ✅ Found
  docs         : ✅ Found
  Base path : /content/drive/MyDrive/MCL_Orchestrator/

✅ All folders confirmed — DRIVE_PATH is locked in


In [ ]:
# Test scikit-fem import and confirm version
try:
    import skfem
    from skfem import MeshQuad, Basis, ElementTriP1, ElementVector
    from skfem import asm, solve, condense
    from skfem.models.elasticity import linear_elasticity

    print("✅ scikit-fem imported successfully")
    print(f"   Version : {skfem.__version__}")
    print(f"   Key modules confirmed:")
    print(f"     - MeshQuad      : mesh generation")
    print(f"     - Basis         : element basis functions")
    print(f"     - ElementTriP1  : linear triangular elements")
    print(f"     - ElementVector : vector field elements")
    print(f"     - asm           : matrix assembly")
    print(f"     - solve         : linear solver")
    print(f"     - condense      : boundary condition handler")

except ImportError as e:
    print(f"❌ scikit-fem import failed: {e}")
    print("   Try: !pip install scikit-fem --upgrade")

✅ scikit-fem imported successfully
   Version : 12.0.1
   Key modules confirmed:
     - MeshQuad      : mesh generation
     - Basis         : element basis functions
     - ElementTriP1  : linear triangular elements
     - ElementVector : vector field elements
     - asm           : matrix assembly
     - solve         : linear solver
     - condense      : boundary condition handler


In [ ]:
# Define log file path
log_file_path = f'{DRIVE_PATH}logs/pipeline.log'

# Explicitly create the log file on Drive first
with open(log_file_path, 'w') as f:
    f.write(f"MCL Orchestrator — Pipeline Log\n")
    f.write(f"Created : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 60 + "\n")

# Confirm file exists before configuring logger
if not os.path.exists(log_file_path):
    print("❌ Could not create log file — check DRIVE_PATH")
else:
    print(f"✅ Log file created at:\n   {log_file_path}")

# Now configure logger to write to that file
logger = logging.getLogger('MCL_Orchestrator')
logger.setLevel(logging.INFO)

# Remove any existing handlers to avoid duplicates on re-run
if logger.handlers:
    logger.handlers.clear()

# File handler — writes to Drive
file_handler = logging.FileHandler(log_file_path)
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter(
    '%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
))

# Console handler — prints to Colab output
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(logging.Formatter(
    '%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
))

logger.addHandler(file_handler)
logger.addHandler(console_handler)

# Define log_event function using named logger
def log_event(stage, status, detail=''):
    message = f"{stage:<30} | {status:<10} | {detail}"
    if status == 'ERROR':
        logger.error(message)
    elif status == 'WARNING':
        logger.warning(message)
    else:
        logger.info(message)

# Test the logger
log_event('00_setup', 'INFO', 'Logger initialised — MCL Orchestrator startup')

# Read back from Drive to confirm
print(f"\n   Reading back from Drive:")
print("   " + "-" * 55)
with open(log_file_path, 'r') as f:
    for line in f.readlines():
        print(f"   {line.strip()}")
print("   " + "-" * 55)
print("\n✅ Logger confirmed working")

2026-06-01 16:21:25 | INFO | 00_setup                       | INFO       | Logger initialised — MCL Orchestrator startup
INFO:MCL_Orchestrator:00_setup                       | INFO       | Logger initialised — MCL Orchestrator startup
2026-06-01 16:21:25 | INFO | 00_setup                       | INFO       | Logger initialised — MCL Orchestrator startup


✅ Log file created at:
   /content/drive/MyDrive/MCL_Orchestrator/logs/pipeline.log

   Reading back from Drive:
   -------------------------------------------------------
   MCL Orchestrator — Pipeline Log
   Created : 2026-06-01 16:21:25
   2026-06-01 16:21:25 | INFO | 00_setup                       | INFO       | Logger initialised — MCL Orchestrator startup
   -------------------------------------------------------

✅ Logger confirmed working


In [ ]:
# Pipeline state dictionary — tracks every stage
pipeline_state = {
    'job_id'      : None,
    'started_at'  : None,
    'stages'      : {
        'A_parameter_generation': 'pending',
        'B_fem_simulation'      : 'pending',
        'C_postprocessing'      : 'pending'
    },
    'errors'      : [],
    'completed_at': None
}

# State file path on Drive
state_file_path = f'{DRIVE_PATH}logs/pipeline_state.json'

def update_state(stage, status, error=None):
    # Update the stage status
    pipeline_state['stages'][stage] = status

    # Log any errors
    if error:
        pipeline_state['errors'].append({
            'stage'    : stage,
            'error'    : str(error),
            'timestamp': datetime.now().isoformat()
        })
        log_event(stage, 'ERROR', str(error))
    else:
        log_event(stage, status, f"Stage updated to {status}")

    # Save full state to Drive
    with open(state_file_path, 'w') as f:
        json.dump(pipeline_state, f, indent=4)

# Test — set a dummy stage to complete
update_state('A_parameter_generation', 'complete')

# Read back from Drive to confirm
print("✅ Pipeline state tracker defined")
print(f"   State file : {state_file_path}")
print(f"\n   Current state:")
print("   " + "-" * 45)
with open(state_file_path, 'r') as f:
    state = json.load(f)
    print(f"   Job ID     : {state['job_id']}")
    print(f"   Stages:")
    for stage, status in state['stages'].items():
        icon = "✅" if status == 'complete' else "⏳"
        print(f"     {icon} {stage:<35} : {status}")
print("   " + "-" * 45)

# Reset state back to pending for actual pipeline use
pipeline_state['stages']['A_parameter_generation'] = 'pending'
print("\n   State reset to pending — ready for pipeline")

2026-06-01 16:42:34 | INFO | A_parameter_generation         | complete   | Stage updated to complete
INFO:MCL_Orchestrator:A_parameter_generation         | complete   | Stage updated to complete
2026-06-01 16:42:34 | INFO | A_parameter_generation         | complete   | Stage updated to complete


✅ Pipeline state tracker defined
   State file : /content/drive/MyDrive/MCL_Orchestrator/logs/pipeline_state.json

   Current state:
   ---------------------------------------------
   Job ID     : None
   Stages:
     ✅ A_parameter_generation              : complete
     ⏳ B_fem_simulation                    : pending
     ⏳ C_postprocessing                    : pending
   ---------------------------------------------

   State reset to pending — ready for pipeline


In [ ]:
def post_with_retry(url, data, max_retries=3, backoff=2):
    """
    Sends a POST request with retry logic and exponential backoff.
    Used by Service A to trigger Service B, and Service B to trigger Service C.

    Args:
        url         : target endpoint URL
        data        : JSON payload to send
        max_retries : number of attempts before giving up
        backoff     : base seconds for exponential wait (2^attempt)

    Returns:
        Parsed JSON response on success

    Raises:
        RuntimeError if all retries are exhausted or timeout occurs
    """
    log_event('retry_handler', 'INFO',
              f"POST to {url} | max_retries={max_retries}")

    for attempt in range(max_retries):
        try:
            response = requests.post(url, json=data, timeout=30)
            response.raise_for_status()
            log_event('retry_handler', 'SUCCESS',
                      f"POST succeeded on attempt {attempt + 1}")
            return response.json()

        except requests.exceptions.ConnectionError:
            if attempt < max_retries - 1:
                wait = backoff ** attempt
                print(f"  ⚠️  Service unreachable — retrying in {wait}s "
                      f"(attempt {attempt + 1}/{max_retries})")
                log_event('retry_handler', 'WARNING',
                          f"Connection failed — retrying in {wait}s")
                time.sleep(wait)
            else:
                log_event('retry_handler', 'ERROR',
                          f"Service at {url} unreachable after {max_retries} attempts")
                raise RuntimeError(
                    f"❌ Service at {url} unreachable after {max_retries} attempts"
                )

        except requests.exceptions.Timeout:
            log_event('retry_handler', 'ERROR',
                      f"Service at {url} timed out after 30s")
            raise RuntimeError(
                f"❌ Service at {url} timed out after 30 seconds"
            )

        except requests.exceptions.HTTPError as e:
            log_event('retry_handler', 'ERROR',
                      f"HTTP error from {url}: {e}")
            raise RuntimeError(f"❌ HTTP error: {e}")

# Test with deliberately unreachable URL
print("Testing retry logic with unreachable URL:")
print("-" * 45)
try:
    post_with_retry(
        url         = 'http://localhost:9999/nonexistent',
        data        = {'test': 'payload'},
        max_retries = 3,
        backoff     = 1
    )
except RuntimeError as e:
    print(f"\n{e}")
    print("-" * 45)
    print("✅ Retry logic confirmed — failed gracefully after 3 attempts")

2026-06-01 16:43:13 | INFO | retry_handler                  | INFO       | POST to http://localhost:9999/nonexistent | max_retries=3
INFO:MCL_Orchestrator:retry_handler                  | INFO       | POST to http://localhost:9999/nonexistent | max_retries=3
2026-06-01 16:43:13 | INFO | retry_handler                  | INFO       | POST to http://localhost:9999/nonexistent | max_retries=3
2026-06-01 16:43:13 | WARNING | retry_handler                  | WARNING    | Connection failed — retrying in 1s
2026-06-01 16:43:13 | WARNING | retry_handler                  | WARNING    | Connection failed — retrying in 1s


Testing retry logic with unreachable URL:
---------------------------------------------
  ⚠️  Service unreachable — retrying in 1s (attempt 1/3)


2026-06-01 16:43:14 | WARNING | retry_handler                  | WARNING    | Connection failed — retrying in 1s
2026-06-01 16:43:14 | WARNING | retry_handler                  | WARNING    | Connection failed — retrying in 1s


  ⚠️  Service unreachable — retrying in 1s (attempt 2/3)


2026-06-01 16:43:15 | ERROR | retry_handler                  | ERROR      | Service at http://localhost:9999/nonexistent unreachable after 3 attempts
ERROR:MCL_Orchestrator:retry_handler                  | ERROR      | Service at http://localhost:9999/nonexistent unreachable after 3 attempts
2026-06-01 16:43:15 | ERROR | retry_handler                  | ERROR      | Service at http://localhost:9999/nonexistent unreachable after 3 attempts



❌ Service at http://localhost:9999/nonexistent unreachable after 3 attempts
---------------------------------------------
✅ Retry logic confirmed — failed gracefully after 3 attempts


In [ ]:
import skfem

print("=" * 55)
print("  MCL ORCHESTRATOR — ENVIRONMENT READY")
print("=" * 55)
print(f"  Timestamp     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n  Libraries:")
print(f"    FastAPI       : confirmed")
print(f"    uvicorn       : confirmed")
print(f"    scikit-fem    : {skfem.__version__}")
print(f"    nest-asyncio  : confirmed")
print(f"    requests      : confirmed")
print(f"\n  Drive Paths:")
print(f"    Base          : {DRIVE_PATH}")
for name, path in PATHS.items():
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"    {name:<12}  : {exists} {path}")
print(f"\n  Shared Utilities:")
print(f"    Logger        : ✅ writing to logs/pipeline.log")
print(f"    State tracker : ✅ writing to logs/pipeline_state.json")
print(f"    Retry handler : ✅ 3 retries with exponential backoff")
print("=" * 55)
print("  ✅ Ready to proceed to Notebook 2 — Service A")
print("=" * 55)

# Write final entry to log
log_event('00_setup', 'INFO',
          'Environment setup complete — all utilities confirmed')

2026-06-01 16:43:43 | INFO | 00_setup                       | INFO       | Environment setup complete — all utilities confirmed
INFO:MCL_Orchestrator:00_setup                       | INFO       | Environment setup complete — all utilities confirmed
2026-06-01 16:43:43 | INFO | 00_setup                       | INFO       | Environment setup complete — all utilities confirmed


  MCL ORCHESTRATOR — ENVIRONMENT READY
  Timestamp     : 2026-06-01 16:43:43

  Libraries:
    FastAPI       : confirmed
    uvicorn       : confirmed
    scikit-fem    : 12.0.1
    nest-asyncio  : confirmed
    requests      : confirmed

  Drive Paths:
    Base          : /content/drive/MyDrive/MCL_Orchestrator/
    notebooks     : ✅ /content/drive/MyDrive/MCL_Orchestrator/notebooks/
    services      : ✅ /content/drive/MyDrive/MCL_Orchestrator/services/
    results       : ✅ /content/drive/MyDrive/MCL_Orchestrator/results/
    logs          : ✅ /content/drive/MyDrive/MCL_Orchestrator/logs/
    docs          : ✅ /content/drive/MyDrive/MCL_Orchestrator/docs/

  Shared Utilities:
    Logger        : ✅ writing to logs/pipeline.log
    State tracker : ✅ writing to logs/pipeline_state.json
    Retry handler : ✅ 3 retries with exponential backoff
  ✅ Ready to proceed to Notebook 2 — Service A
